<a href="https://colab.research.google.com/github/jigneshpandya86-lab/ALMWork/blob/main/MyAlembic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install python-docx python-pptx openpyxl pillow pandas xlrd graphviz reportlab pandas networkx openpyxl

In [5]:
import pandas as pd
import re
import io
from google.colab import files

def process_excel_batches():
    print("Please upload your Batch Excel (.xlsx) files...")
    # This will open a file upload dialog in Colab
    uploaded = files.upload()

    processed_dfs = []
    batch_names = []

    # Iterate through all uploaded files
    for filename, content in uploaded.items():
        if not filename.endswith(('.xlsx', '.xls', '.csv')):
            print(f"Skipping {filename}: Not an Excel or CSV file.")
            continue

        # 1. Extract the Batch Number from the filename
        match = re.search(r'Batch No\.? #\s*(\d+)', filename)
        if match:
            batch_no = match.group(1)
        else:
            batch_no = filename.split('.')[0]

        # 2. Read the file into a DataFrame without headers
        try:
            # Added slight fallback in case you accidentally upload CSVs
            if filename.endswith('.csv'):
                df_raw = pd.read_csv(io.BytesIO(content), header=None, on_bad_lines='skip')
            else:
                df_raw = pd.read_excel(io.BytesIO(content), header=None)
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            continue

        # 3. Find ALL start and end indices for every page in the sheet
        start_indices = []
        end_indices = []

        for i, row in df_raw.iterrows():
            row_text = str(row.values)

            # Record every time we see a header
            if 'DATE & TIME' in row_text:
                start_indices.append(i + 2) # Skip the unit row below it

            # Record every time we see a footer, making sure it corresponds to an open header
            elif 'Report Generated' in row_text:
                if len(start_indices) > len(end_indices):
                    end_indices.append(i)

        # In case the very last page is missing a footer, cap it at the end of the file
        if len(start_indices) > len(end_indices):
            end_indices.append(len(df_raw))

        # 4. Extract all data chunks and stitch them vertically
        file_chunks = []
        for s, e in zip(start_indices, end_indices):
            # Extract just Date/Time and Temp columns for this specific page
            chunk = df_raw.iloc[s:e, 0:2].copy()
            chunk.dropna(how='all', inplace=True)
            file_chunks.append(chunk)

        if file_chunks:
            # Combine all the pages of this batch into one continuous vertical dataset
            batch_data = pd.concat(file_chunks, ignore_index=True)
            # Standardize column names so pandas can combine multiple batches later
            batch_data.columns = [0, 1]

            processed_dfs.append(batch_data)
            batch_names.append(f"BATCH No : {batch_no}")
        else:
            print(f"Could not find any 'DATE & TIME' headers in {filename}. Skipping.")

    # 5. Combine everything and format the final file
    if processed_dfs:
        # Concatenate all different batches side-by-side
        data_df = pd.concat(processed_dfs, axis=1, ignore_index=True)

        # Create two manual header rows
        header_row_1 = []
        header_row_2 = []

        for batch in batch_names:
            header_row_1.extend([batch, ""]) # Blank space to mimic merged cell
            header_row_2.extend(["DATE & TIME", "°C"])

        top_rows = pd.DataFrame([header_row_1, header_row_2])
        top_rows.columns = data_df.columns

        # Stack headers on top of data
        final_df = pd.concat([top_rows, data_df], ignore_index=True)

        # 6. Save and Download
        output_filename = 'Desired_Output.xlsx'
        final_df.to_excel(output_filename, index=False, header=False)

        print(f"\n✅ Processing complete! Found {len(start_indices)} pages for {filename}. Downloading {output_filename}...")
        files.download(output_filename)
    else:
        print("\n❌ No valid data was found to process.")

# Run the function
process_excel_batches()

Please upload your Batch Excel (.xlsx) files...


Saving Batch No. # 2502000159.xlsx to Batch No. # 2502000159 (3).xlsx
Saving Batch No. # 2502006570.xlsx to Batch No. # 2502006570 (3).xlsx
Saving Batch No. # 2502006571.xlsx to Batch No. # 2502006571 (3).xlsx
Saving Batch No. # 2502006572.xlsx to Batch No. # 2502006572 (3).xlsx
Saving Batch No. # 2502006574.xlsx to Batch No. # 2502006574 (3).xlsx
Saving Batch No. # 2502006575.xlsx to Batch No. # 2502006575 (3).xlsx
Saving Batch No. # 2502006576.xlsx to Batch No. # 2502006576 (3).xlsx
Saving Batch No. # 2502006577.xlsx to Batch No. # 2502006577 (3).xlsx
Saving Batch No. # 2502006578.xlsx to Batch No. # 2502006578 (3).xlsx
Saving Batch No. # 2502006579.xlsx to Batch No. # 2502006579 (3).xlsx
Saving Batch No. # 2502006580.xlsx to Batch No. # 2502006580 (3).xlsx
Saving Batch No. # 2502008643.xlsx to Batch No. # 2502008643 (3).xlsx
Saving Batch No. # 2502008644.xlsx to Batch No. # 2502008644 (3).xlsx
Saving Batch No. # 2502009551.xlsx to Batch No. # 2502009551 (3).xlsx

✅ Processing comple

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>